In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Run_concurrent_tasks")
    .master("local[*]")
    .getOrCreate())

spark

In [2]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
df_cities =spark.read.format("csv").option("header", True).load('/content/drive/MyDrive/cities.csv')

In [4]:
df_cities.show()

+----------+--------------------+-----+---------+--------------------+
|   city_id|                city|state|state_abv|             country|
+----------+--------------------+-----+---------+--------------------+
| 324732552|   ladang ulu bernam| NULL|     NULL|            Malaysia|
|2010991182|        `ali rowshan| NULL|     NULL|                Iran|
| 741912760|              borovo| NULL|     NULL|Bosnia and Herzeg...|
| 604192006|         aillo talor| NULL|     NULL|               Chile|
| 752393249|      sheykheh koreh| NULL|     NULL|                Iran|
| 797426237|          saro harai| NULL|     NULL|            Pakistan|
|1339946600|            lamadihi| NULL|     NULL|               Nepal|
| 216616140|            krokohwe| NULL|     NULL|               Ghana|
|1738243674|      ianakandrarezo| NULL|     NULL|          Madagascar|
|1989452216|               bumay| NULL|     NULL|         Afghanistan|
|1972802272|         bogale-kota| NULL|     NULL|Central African R...|
| 5440

In [5]:
def extract_country_data(_country: str):
    try:
        # Read Cities data
        df_cities = (
            spark
            .read
            .format("csv")
            .option("header", True)
            .load("/content/drive/MyDrive/cities.csv")
        )

        # Fiter data
        df_final = df_cities.where(f"lower(country) = lower('{_country}')")

        # Write data
        (
            df_final
            .coalesce(1)
            .write
            .format("csv")
            .mode("overwrite")
            .option("header", True)
            .save(f"/content/drive/MyDrive/countries/{_country.lower()}/")
        )

        return f"Data Extracted for {_country} at: [/content/drive/MyDrive/countries/{_country.lower()}/]"
    except Exception as e:
        raise Exception(e)

In [7]:
# Use For loops to execute the jobs
import time

# Set start time
start_time = time.time()

# Run all extracts through for-loop
_countries = ['India', 'Iran', 'Ghana', 'Australia']

for _country in _countries:
    print(extract_country_data(_country))

# End time
end_time = time.time()

# Total time taken
print(f"total time = {end_time - start_time} seconds")

Data Extracted for India at: [/content/drive/MyDrive/countries/india/]
Data Extracted for Iran at: [/content/drive/MyDrive/countries/iran/]
Data Extracted for Ghana at: [/content/drive/MyDrive/countries/ghana/]
Data Extracted for Australia at: [/content/drive/MyDrive/countries/australia/]
total time = 29.03846311569214 seconds


In [8]:
# Use threads to run the queries in concurrently/parallely
import time
import concurrent.futures

# Set start time
start_time = time.time()

_countries = ['India', 'Iran', 'Ghana', 'Australia']

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    results = {executor.submit(extract_country_data, _country) for _country in _countries}

    for result in results:
        print(result.result())

# End time
end_time = time.time()

# Total time taken
print(f"total time = {end_time - start_time} seconds")

Data Extracted for India at: [/content/drive/MyDrive/countries/india/]
Data Extracted for Ghana at: [/content/drive/MyDrive/countries/ghana/]
Data Extracted for Australia at: [/content/drive/MyDrive/countries/australia/]
Data Extracted for Iran at: [/content/drive/MyDrive/countries/iran/]
total time = 18.0979266166687 seconds
